# Exploratory Transcriptomic Analysis of Inflammatory Patterns in Psoriasis-Associated Adipose Tissue


## Project Overview

This project explores a public RNA-seq dataset of cutaneous adipose tissue to examine how transcriptomic patterns differ across healthy control, lesional psoriasis, and non-lesional psoriasis samples. The analysis focuses on sample grouping, exploratory gene expression analysis, and visualisation of inflammatory patterns in psoriasis-associated adipose tissue.


### Research Question
How do transcriptomic patterns differ between healthy control, lesional psoriasis, and non-lesional psoriasis adipose tissue samples?


## 1. Import Required Libraries


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load and Inspect the Data Files

In [ ]:
print(os.listdir())

In [ ]:
counts = pd.read_csv("GSE287022_raw_counts_per_gene.csv")
counts.head()

In [ ]:
counts.shape

In [ ]:
counts.columns[:10]

In [ ]:
counts.iloc[:5, :5]

## 3. Extract and Clean Sample Metadata

In [ ]:
with open("GSE287022_series_matrix.txt", "r") as f:
    lines = f.readlines()

sample_lines = [line.strip() for line in lines if line.startswith("!Sample")]

for line in sample_lines[:25]:
    print(line)

In [ ]:
sample_accession_line = [line for line in sample_lines if line.startswith("!Sample_geo_accession")][0]
sample_title_line = [line for line in sample_lines if line.startswith("!Sample_title")][0]
sample_source_line = [line for line in sample_lines if line.startswith("!Sample_source_name_ch1")][0]
sample_characteristics = [line for line in sample_lines if line.startswith("!Sample_characteristics_ch1")]

In [ ]:
sample_ids = sample_accession_line.split("\t")[1:]
sample_titles = sample_title_line.split("\t")[1:]
sample_sources = sample_source_line.split("\t")[1:]

metadata = pd.DataFrame({
    "Sample_ID": sample_ids,
    "Title": sample_titles,
    "Source": sample_sources
})

for i, line in enumerate(sample_characteristics):
    values = line.split("\t")[1:]
    metadata[f"Characteristic_{i+1}"] = values

metadata.head()

In [ ]:
metadata_clean = metadata.copy()

str_cols = metadata_clean.select_dtypes(include="object").columns
metadata_clean[str_cols] = metadata_clean[str_cols].apply(lambda col: col.str.strip('"'))

metadata_clean["Condition"] = metadata_clean["Characteristic_2"].str.replace("condition: ", "", regex=False)
metadata_clean["Timepoint"] = metadata_clean["Characteristic_3"].str.replace("timepoint: ", "", regex=False)
metadata_clean["Tissue"] = metadata_clean["Characteristic_4"].str.replace("tissue: ", "", regex=False)
metadata_clean["Treatment"] = metadata_clean["Characteristic_5"].str.replace("treatment: ", "", regex=False)
metadata_clean["Subject_ID"] = metadata_clean["Characteristic_6"].str.replace("subjectid: ", "", regex=False)

metadata_clean[["Sample_ID", "Title", "Condition", "Timepoint", "Tissue", "Treatment", "Subject_ID"]].head(10)

Sample metadata were extracted from the GEO series matrix file and cleaned to identify biologically meaningful variables such as condition, tissue type, timepoint, and treatment.

## 4. Define Analysis Groups

In [ ]:
metadata_filtered = metadata_clean[
    (
        (metadata_clean["Condition"] == "Healthy Vol.") &
        (metadata_clean["Tissue"] == "Normal")
    )
    |
    (
        (metadata_clean["Condition"] == "Psoriasis") &
        (metadata_clean["Timepoint"] == "Baseline") &
        (metadata_clean["Tissue"].isin(["Lesional", "Non-Lesional"]))
    )
].copy()

metadata_filtered[["Sample_ID", "Condition", "Timepoint", "Tissue", "Treatment"]].head(15)

In [ ]:
def assign_group(row):
    if row["Condition"] == "Healthy Vol." and row["Tissue"] == "Normal":
        return "Healthy_Control"
    elif row["Condition"] == "Psoriasis" and row["Tissue"] == "Lesional" and row["Timepoint"] == "Baseline":
        return "Psoriasis_Lesional_Baseline"
    elif row["Condition"] == "Psoriasis" and row["Tissue"] == "Non-Lesional" and row["Timepoint"] == "Baseline":
        return "Psoriasis_NonLesional_Baseline"
    else:
        return "Other"

metadata_filtered["Group"] = metadata_filtered.apply(assign_group, axis=1)
metadata_filtered["Group"].value_counts()

For this exploratory analysis, samples were restricted to healthy controls and baseline psoriasis samples (lesional and non-lesional). This avoids confounding from treatment and follow-up timepoints and allows a cleaner comparison of transcriptomic patterns.

## 5. Prepare the Counts Matrix

In [ ]:
counts.columns = [col.strip('"') if isinstance(col, str) else col for col in counts.columns]
counts.columns[:10]

In [ ]:
matched_samples = list(set(metadata_filtered["Sample_ID"]).intersection(set(counts.columns)))
len(matched_samples)

In [ ]:
matched_samples[:10]

In [ ]:
num_count_samples = len(counts.columns) - 1   # minus the first gene column
num_metadata_rows = len(metadata_clean)

print("Number of sample columns in counts:", num_count_samples)
print("Number of rows in metadata:", num_metadata_rows)

In [ ]:
sample_cols = list(counts.columns[1:])   # skip the first column, which is genes
metadata_clean["Count_Column"] = sample_cols

metadata_clean[["Sample_ID", "Title", "Count_Column", "Condition", "Timepoint", "Tissue"]].head(10)

In [ ]:
metadata_filtered = metadata_clean[
    (
        (metadata_clean["Condition"] == "Healthy Vol.") &
        (metadata_clean["Tissue"] == "Normal")
    )
    |
    (
        (metadata_clean["Condition"] == "Psoriasis") &
        (metadata_clean["Timepoint"] == "Baseline") &
        (metadata_clean["Tissue"].isin(["Lesional", "Non-Lesional"]))
    )
].copy()

In [ ]:
def assign_group(row):
    if row["Condition"] == "Healthy Vol." and row["Tissue"] == "Normal":
        return "Healthy_Control"
    elif row["Condition"] == "Psoriasis" and row["Tissue"] == "Lesional" and row["Timepoint"] == "Baseline":
        return "Psoriasis_Lesional_Baseline"
    elif row["Condition"] == "Psoriasis" and row["Tissue"] == "Non-Lesional" and row["Timepoint"] == "Baseline":
        return "Psoriasis_NonLesional_Baseline"
    else:
        return "Other"

metadata_filtered["Group"] = metadata_filtered.apply(assign_group, axis=1)
metadata_filtered["Group"].value_counts()

In [ ]:
matched_samples = metadata_filtered["Count_Column"].tolist()
len(matched_samples), matched_samples[:10]

In [ ]:
gene_col = counts.columns[0]
counts_subset = counts[[gene_col] + matched_samples].copy()
counts_subset.head()

## 6. Exploratory Transcriptomic Analysis

This section focuses on exploratory analysis of the transcriptomic data. To make the count data easier to interpret, the selected expression matrix is prepared, transformed, and used to identify genes with the greatest variability across the sample groups.

In [ ]:
counts_subset.head()

In [ ]:
gene_col = counts_subset.columns[0]
print("Gene column name:", gene_col)
print(counts_subset[gene_col].head())

In [ ]:
expression = counts_subset.set_index(gene_col)
expression.head()

In [ ]:
expression.shape

In [ ]:
expression = expression.apply(pd.to_numeric, errors="coerce")
expression.dtypes.head()

In [ ]:
expression.isnull().sum().sum()

In [ ]:
expression_log = np.log1p(expression)
expression_log.head()

The raw counts were log-transformed using log1p to reduce skewness and make expression patterns easier to compare visually across samples.

In [ ]:
metadata_filtered["Group"].value_counts()

In [ ]:
group_order = [
    "Healthy_Control",
    "Psoriasis_NonLesional_Baseline",
    "Psoriasis_Lesional_Baseline"
]

metadata_filtered["Group"] = pd.Categorical(metadata_filtered["Group"], categories=group_order, ordered=True)
metadata_filtered = metadata_filtered.sort_values("Group")
metadata_filtered[["Sample_ID", "Count_Column", "Group"]].head(15)

In [ ]:
ordered_samples = metadata_filtered["Count_Column"].tolist()
ordered_samples[:10]

In [ ]:
expression_log = expression_log[ordered_samples]
expression_log.head()

In [ ]:
gene_variance = expression_log.var(axis=1)
top_variable_genes = gene_variance.sort_values(ascending=False).head(30).index
top_variable_genes

In [ ]:
top_expression = expression_log.loc[top_variable_genes]
top_expression.head()

In [ ]:
top_expression_named = top_expression.copy()
top_expression_named.index = [
    id_to_symbol.get(gene_id, gene_id) for gene_id in top_expression_named.index
]

plt.figure(figsize=(14,8))
sns.heatmap(top_expression_named, cmap="viridis", yticklabels=True, xticklabels=False)
plt.title("Top 30 Most Variable Genes Across Selected Samples")
plt.xlabel("Samples")
plt.ylabel("Genes")
plt.show()

The heatmap of the most variable genes provides an initial view of transcriptomic heterogeneity across the selected sample groups. Differences in expression patterns across healthy control, non-lesional psoriasis, and lesional psoriasis samples may reflect biologically meaningful inflammatory variation.

## 7. Visualisation

This section uses dimensionality reduction and grouped visualisation to explore whether healthy control, non-lesional psoriasis, and lesional psoriasis samples show distinct transcriptomic patterns.

In [ ]:
metadata_plot = metadata_filtered.set_index("Count_Column").loc[ordered_samples].reset_index()
metadata_plot[["Count_Column", "Group"]].head()

In [ ]:
pca_input = top_expression.T
pca_input.shape

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(pca_input)

pca_df = pd.DataFrame(pca_coords, columns=["PC1", "PC2"], index=pca_input.index).reset_index()
pca_df = pca_df.rename(columns={"index": "Count_Column"})
pca_df = pca_df.merge(metadata_plot[["Count_Column", "Group"]], on="Count_Column")

pca_df.head()

In [ ]:
pca.explained_variance_ratio_

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Group", s=70)
plt.title("PCA of Top Variable Genes Across Selected Samples")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.show()

The PCA plot provides a low-dimensional view of transcriptomic variation across the selected samples. PC1 explained 82.0% of the variance and PC2 explained 11.8%, suggesting that a large proportion of overall variation was captured in the first two components. While the three groups showed partial overlap, the distribution of samples indicates meaningful transcriptomic heterogeneity across healthy control, non-lesional psoriasis, and lesional psoriasis adipose tissue.

In [ ]:
counts = counts.rename(columns={counts.columns[0]: "Ensembl_ID"})

In [ ]:
metadata_filtered["Group_Short"] = metadata_filtered["Group"].replace({
    "Healthy_Control": "Healthy",
    "Psoriasis_NonLesional_Baseline": "NonLesional",
    "Psoriasis_Lesional_Baseline": "Lesional"
})

metadata_filtered[["Group", "Group_Short"]].drop_duplicates()

In [ ]:
group_map_short = metadata_filtered.set_index("Count_Column")["Group_Short"]

group_mean_expression_short = (
    expression_log[ordered_samples]
    .T
    .join(group_map_short.rename("Group"))
    .groupby("Group")
    .mean()
    .T
)

group_mean_expression_short.head()

In [ ]:
top_group_expression_short = group_mean_expression_short.loc[top_variable_genes]
top_group_expression_short.head()

In [ ]:
!pip install mygene

In [ ]:
import mygene

mg = mygene.MyGeneInfo()

In [ ]:
top_gene_ids = list(top_variable_genes)
top_gene_ids[:10]

In [ ]:
gene_info = mg.querymany(
    top_gene_ids,
    scopes="ensembl.gene",
    fields="symbol,name",
    species="human"
)

gene_info[:5]

In [ ]:
mapping_df = pd.DataFrame(gene_info)[["query", "symbol", "name"]]
mapping_df

In [ ]:
id_to_symbol = dict(zip(mapping_df["query"], mapping_df["symbol"]))
id_to_symbol

In [ ]:
top_group_expression_short = group_mean_expression_short.loc[top_variable_genes].copy()

top_group_expression_named = top_group_expression_short.copy()
top_group_expression_named.index = [
    id_to_symbol.get(gene_id, gene_id) for gene_id in top_group_expression_named.index
]

top_group_expression_named.head()

In [ ]:
plt.figure(figsize=(8,10))
sns.heatmap(top_group_expression_named, cmap="viridis", linewidths=0.2)
plt.title("Average Expression of Top Variable Genes by Group")
plt.xlabel("Group")
plt.ylabel("Genes")
plt.tight_layout()
plt.show()

The group-average heatmap provides a clearer summary of transcriptomic differences across Healthy, NonLesional, and Lesional adipose tissue samples. Several highly variable genes showed distinct average expression patterns across the three groups. While some genes appeared elevated in both psoriasis-associated groups relative to healthy controls, others differed between non-lesional and lesional tissue, suggesting biologically meaningful heterogeneity within psoriasis-associated adipose tissue.

In [ ]:
mapping_df[mapping_df["symbol"].isna()]

## 8. Key Findings

## Key Findings

- The selected adipose tissue samples showed clear transcriptomic
variability across healthy control, non-lesional psoriasis, and lesional psoriasis groups.
- The PCA captured most of the overall variation within the first two components, suggesting that major expression differences were reflected in the reduced-dimensional view.
- The grouped heatmap showed that several highly variable genes differed in their average expression across the three groups, pointing towards meaningful biological differences in the tissue states.
- Non-lesional and lesional psoriasis-associated adipose tissue did not appear transcriptomically identical, which suggests that even tissue not visibly lesional may still carry distinct molecular changes.
- Overall, this project helped me understand how public RNA-seq data can be organised, filtered, transformed, and visualised to explore transcriptomic patterns in a clinically relevant disease context.